In [1]:
import geopandas as gpd
import math
from logging import getLogger

import numpy as np
import pandas as pd
import shapely
from shapely import LineString
from shapely.geometry import Polygon, MultiPolygon, LineString, Point
from shapely.ops import unary_union, transform, nearest_points
import geopandas as gpd
import pyproj

In [2]:
file_path = '/Users/dalo2903/repos/ics209-plus-fired/output/linked_nirops_fired.gpkg'
# --- Read layers ---
gdf_nirops = gpd.read_file(file_path, layer='nirops')
gdf_fired = gpd.read_file(file_path, layer='fired')

# --- Ensure CRS is projected (in meters) ---
if gdf_nirops.crs.is_geographic:
    gdf_nirops = gdf_nirops.to_crs("EPSG:5070")
if gdf_fired.crs.is_geographic:
    gdf_fired = gdf_fired.to_crs("EPSG:5070")

In [6]:
# --- Process NIROPS events ---
nirops_results_list = []

for event_name, group in gdf_nirops.groupby('NIROPS_Incident_C'):
    print(event_name)
    # Remove rows with duplicate NIROPS_UTC values
    group = group.drop_duplicates(subset='NIROPS_UTC')

    # Run computefirespeed on this event's subset
    result_gdf = computefirespeed(group, id_col='NIROPS_Incident_C')
    nirops_results_list.append(result_gdf)

# Concatenate all processed NIROPS events back into one GeoDataFrame
gdf_nirops_processed = gpd.GeoDataFrame(pd.concat(nirops_results_list, ignore_index=True))

AZ-A3S-001033_Walnut
AZ-A4S-000950_Central
AZ-A5S-230970_Pilot
AZ-ASD-000097_Basin
AZ-ASF-000613_Horton
AZ-ASF-000624_Bear
Time step 7, child index 0
Child geometry area: 96529891.52558418
Parent geometries areas: [95930545.2271926]
Intersection matrix:
[[ True]]
AZ-ASF-000662_Wyrick
Time step 2, child index 0
Child geometry area: 30749411.551032986
Parent geometries areas: [30749411.551032986]
Intersection matrix:
[[ True]]
AZ-ASF-00083_Bringham
Time step 4, child index 0
Child geometry area: 59189709.10126071
Parent geometries areas: [6673906.47475449]
Intersection matrix:
[[ True]]
Time step 5, child index 0
Child geometry area: 62317507.94476351
Parent geometries areas: [59189709.10126071]
Intersection matrix:
[[ True]]
Time step 7, child index 0
Child geometry area: 72269821.10969272
Parent geometries areas: [70491966.79146272]
Intersection matrix:
[[ True]]
Time step 11, child index 0
Child geometry area: 93574412.8792964
Parent geometries areas: [89990160.3645498]
Intersection m

KeyboardInterrupt: 

In [5]:
# --- Read NIROPS layer ---
gdf_nirops = gpd.read_file(file_path, layer='nirops')

# Run computefirespeed on NIROPS geometries
nirops_results = gdf_nirops['geometry'].apply(computefirespeed)
gdf_nirops['NIROPS_orig_x'] = nirops_results.apply(lambda x: x[0])
gdf_nirops['NIROPS_orig_y'] = nirops_results.apply(lambda x: x[1])
gdf_nirops['NIROPS_dest_x'] = nirops_results.apply(lambda x: x[2])
gdf_nirops['NIROPS_dest_y'] = nirops_results.apply(lambda x: x[3])
gdf_nirops['NIROPS_result_max_dist'] = nirops_results.apply(lambda x: x[4])
gdf_nirops['NIROPS_result_speed'] = nirops_results.apply(lambda x: x[5])

# --- Read FIRED layer ---
gdf_fired = gpd.read_file(file_path, layer='fired')

# Run computefirespeed on FIRED geometries
fired_results = gdf_fired['geometry'].apply(computefirespeed)
gdf_fired['FIRED_orig_x'] = fired_results.apply(lambda x: x[0])
gdf_fired['FIRED_orig_y'] = fired_results.apply(lambda x: x[1])
gdf_fired['FIRED_dest_x'] = fired_results.apply(lambda x: x[2])
gdf_fired['FIRED_dest_y'] = fired_results.apply(lambda x: x[3])
gdf_fired['FIRED_result_max_dist'] = fired_results.apply(lambda x: x[4])
gdf_fired['FIRED_result_speed'] = fired_results.apply(lambda x: x[5])

# Optional: merge results back together on a shared identifier if needed
# gdf_merged = gdf_nirops.merge(gdf_fired, on='shared_id_column')

print(gdf_nirops.head())
print(gdf_fired.head())


AttributeError: 'MultiPolygon' object has no attribute 'crs'

In [ ]:
# --- Process FIRED events ---
fired_results_list = []

for event_name, group in gdf_fired.groupby('FIRED_id'):
    # Run computefirespeed on this event's subset
    result_gdf = computefirespeed(group, id_col='FIRED_id')
    fired_results_list.append(result_gdf)

# Concatenate all processed FIRED events back into one GeoDataFrame
gdf_fired_processed = gpd.GeoDataFrame(pd.concat(fired_results_list, ignore_index=True))


In [3]:
gdf_nirops

,NIROPS_Incident_C,NIROPS_UTC,NIROPS_DateUTC,NIROPS_Acres,NIROPS_IRWINID,NIROPS_Inc Name,NIROPS_Inc Number,NIROPS_perim_date,FIRED_did,FIRED_id,...,FIRED_lc_code,FIRED_lc_mode,FIRED_lc_name,FIRED_lc_desc,FIRED_lc_type,FIRED_eco_mode,FIRED_eco_name,FIRED_eco_type,FIRED_perim_date,geometry
0,AZ-ASD-000097_Basin,2020-05-12 00:00:00,2020-05-12,36396.880411,3bab66f5-b2c8-4801-9590-ebe92e0d79c3,Basin,AZ-ASD-000097,2020-05-12,fd6407021591ec2d8ed5ebc12699a1fe,4129,...,7,7,Open Shrublands,Dominated by woody perennials (1-2m height)10-...,IGBP global vegetation classification scheme,4.0,Colorado Plateau shrublands,WWF Terrestrial Ecoregions of the World,2020-05-12,"MULTIPOLYGON (((-1574581.555 1643578.888, -157..."
1,AZ-TNF-001306_Sawtooth,2020-06-02 06:47:00,2020-06-02,20987.284207,457c72ec-7f5b-4b5b-aed4-dc680050bf53,Sawtooth,AZ-TNF-001306,2020-06-02,a29616224a48d7fc0d2134dd5172bbd3,6991,...,7,7,Open Shrublands,Dominated by woody perennials (1-2m height)10-...,IGBP global vegetation classification scheme,3.0,Arizona Mountains forests,WWF Terrestrial Ecoregions of the World,2020-06-02,"MULTIPOLYGON (((-1410740.368 1257427.962, -141..."
2,AZ-TNF-001306_Sawtooth,2020-06-02 06:47:00,2020-06-02,20987.284207,457c72ec-7f5b-4b5b-aed4-dc680050bf53,Sawtooth,AZ-TNF-001306,2020-06-02,a29616224a48d7fc0d2134dd5172bbd3,6991,...,7,7,Open Shrublands,Dominated by woody perennials (1-2m height)10-...,IGBP global vegetation classification scheme,3.0,Arizona Mountains forests,WWF Terrestrial Ecoregions of the World,2020-06-02,"MULTIPOLYGON (((-1410842.96 1257452.413, -1410..."
3,AZ-TNF-001306_Sawtooth,2020-06-02 06:47:00,2020-06-02,20987.284207,457c72ec-7f5b-4b5b-aed4-dc680050bf53,Sawtooth,AZ-TNF-001306,2020-06-02,a29616224a48d7fc0d2134dd5172bbd3,6991,...,7,7,Open Shrublands,Dominated by woody perennials (1-2m height)10-...,IGBP global vegetation classification scheme,3.0,Arizona Mountains forests,WWF Terrestrial Ecoregions of the World,2020-06-02,"MULTIPOLYGON (((-1412131.578 1266938.536, -141..."
4,AZ-TNF-001306_Sawtooth,2020-06-02 06:47:00,2020-06-02,20987.284207,457c72ec-7f5b-4b5b-aed4-dc680050bf53,Sawtooth,AZ-TNF-001306,2020-06-02,a29616224a48d7fc0d2134dd5172bbd3,6991,...,7,7,Open Shrublands,Dominated by woody perennials (1-2m height)10-...,IGBP global vegetation classification scheme,3.0,Arizona Mountains forests,WWF Terrestrial Ecoregions of the World,2020-06-02,"MULTIPOLYGON (((-1411998.649 1266987.507, -141..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8828,AZ-TNF-001315_Valentine,2023-10-01 00:49:00,2023-10-01,9533.819774,f0681379-0285-4155-8129-af4b8e74cece,Valentine,AZ-TNF-001315,2023-10-01,980a6e80eb6eafda24321cc9ec1ad4d2,6135,...,8,8,Woody Savannas,Tree cover 30-60% (canopy>2m).,IGBP global vegetation classification scheme,3.0,Arizona Mountains forests,WWF Terrestrial Ecoregions of the World,2023-10-01,"MULTIPOLYGON (((-1350370.325 1344472.984, -135..."
8829,CO-SJF-001184_TrailSprings,2023-10-24 00:54:00,2023-10-24,725.627107,5664eea7-1f0b-49f2-b588-8e5c80f30a9e,Trail Springs,CO-SJF-001184,2023-10-24,a553363de192268cfa287c6d7ebd283a,51265,...,8,8,Woody Savannas,Tree cover 30-60% (canopy>2m).,IGBP global vegetation classification scheme,11.0,Colorado Rockies forests,WWF Terrestrial Ecoregions of the World,2023-10-24,"MULTIPOLYGON (((-983297.027 1644404.033, -9832..."
8830,CA-RRU-160976_Highland,2023-11-01 04:00:00,2023-11-01,2484.755872,a81913df-822a-4adb-a61f-a3b07b21ffb4,HIGHLAND,CA-RRU-160976,2023-11-01,99ef1222ce590d15fa1c3a9b1f8e8157,6962,...,7,10,Grasslands,Dominated by herbaceous annuals (<2m),IGBP global vegetation classification scheme,1.0,California coastal sage and chaparral,WWF Terrestrial Ecoregions of the World,2023-11-01,"MULTIPOLYGON (((-1914890.717 1366524.057, -191..."
8831,CA-RRU-160976_Highland,2023-11-02 04:00:00,2023-11-02,2492.055492,a81913df-822a-4adb-a61f-a3b07b21ffb4,HIGHLAND,CA-RRU-160976,2023-11-02,6a38599578c74d7368171d2a48cd9aed,6962,...,10,10,Grasslands,Dominated by 

In [4]:
def computefirespeed(fire_gdf, id_col="id"):
    if fire_gdf.crs is None or fire_gdf.crs.is_geographic:
        raise ValueError(
            f"computefirespeed requires projected CRS in meters, got {fire_gdf.crs}"
        )
    transformer = pyproj.Transformer.from_crs(fire_gdf.crs, "EPSG:4326", always_xy=True)
    geod = pyproj.Geod(ellps="WGS84")
    fire_gdf = fire_gdf.reset_index(drop=True).copy()
    n_rows = fire_gdf.shape[0]
    orig_x = [np.nan] * n_rows
    orig_y = [np.nan] * n_rows
    dest_x = [np.nan] * n_rows
    dest_y = [np.nan] * n_rows
    result_max_dist = [np.nan] * n_rows
    result_speed = [np.nan] * n_rows

    has_ids = id_col in fire_gdf.columns

    ### iterate over time steps
    for i in range(1, fire_gdf.shape[0]):
        # first perimeter in each fire has no valid predecessor
        if has_ids and fire_gdf.iloc[i][id_col] != fire_gdf.iloc[i - 1][id_col]:
            continue

        prev_geom = fire_gdf.iloc[i - 1].geometry #replaced 'cum_geom'
        curr_geom = fire_gdf.iloc[i].geometry #replaced 'cum_geom'

        #print("timestep:", i)

        # ensure MultiPolygon
        if isinstance(prev_geom, Polygon):
            prev_geom = MultiPolygon([prev_geom])
        if isinstance(curr_geom, Polygon):
            curr_geom = MultiPolygon([curr_geom])

        # --- parent-child intersection matrix ---
        inter_matrix = np.zeros((len(prev_geom.geoms), len(curr_geom.geoms)),dtype=bool)

        for ii in range(len(prev_geom.geoms)):
            for jj in range(len(curr_geom.geoms)):
                inter_matrix[ii, jj] = (
                    prev_geom.geoms[ii].buffer(1e-6).intersects(curr_geom.geoms[jj]))
            
        # --- parent perimeter coordinates ---
        prev_coords = [
            prev_geom.geoms[ii].simplify(0.05).exterior.coords
            for ii in range(len(prev_geom.geoms))
        ]

        best_dist = -np.inf
        best_origin = None
        best_dest = None

        diag_rows = []   # diagnostics for this timestep
        best_child = None

        # --- LOOP OVER CHILD POLYGONS ---
        for j, child_poly in enumerate(curr_geom.geoms):

            parent_ids = np.where(inter_matrix[:, j])[0].tolist()            
            chosen_parent = None

            # --- spot fire handling ---
            if len(parent_ids) == 0:
                dists = [
                    prev_poly.distance(child_poly)
                    for prev_poly in prev_geom.geoms
                ]
                parent_ids = [int(np.argmin(dists))]

            parent_geoms = [prev_geom.geoms[ii] for ii in parent_ids]
            parent_coords = [prev_coords[ii] for ii in parent_ids]

            dist, origin, dest, parent_local_idx = compute_max_vector(
                perim_inner_geoms=parent_geoms,
                perim_outer_geoms=[child_poly],
                inter_matrix=np.ones((len(parent_geoms), 1)), # fix shape: N x 1
                spot_threshold=4000
            )
            # --- DEBUGGING CHECK ---
            if parent_local_idx is None:
                print("WARNING: No valid parent polygon found for child polygon")
                print(f"Time step {i}, child index {j}")
                print(f"Child geometry area: {child_poly.area}")
                print(f"Parent geometries areas: {[p.area for p in parent_geoms]}")
                print(f"Intersection matrix:\n{inter_matrix}")
                continue  # skip this child polygon
            
            # infer which parent geometry produced the origin
            chosen_parent = parent_ids[parent_local_idx]

            if dist > best_dist:
                best_dist = dist
                best_origin = origin
                best_dest = dest
                best_child = j

        # --- finalize timestep ---
        if best_origin is None:
            continue
        
        orig_x[i] = best_origin[0]
        orig_y[i] = best_origin[1]
        dest_x[i] = best_dest[0]
        dest_y[i] = best_dest[1]

        lons, lats = transformer.transform(
            [best_origin[0], best_dest[0]],
            [best_origin[1], best_dest[1]]
        )

        dist_m = geod.line_length(lons, lats)

        result_max_dist[i] = dist_m / 1000
        result_speed[i] = (dist_m / 1000) / 24

    return (orig_x, orig_y, dest_x, dest_y, result_max_dist, result_speed)


def compute_max_vector(perim_inner_geoms,
                               perim_outer_geoms,
                               inter_matrix,
                               spot_threshold=4000,
                               debug=False):
    
    result_dist = []
    result_coord_pair = []
    result_poly_pair = []
    result_parent_idx = []
    
    points_per_meter = 1 / 200

    for poly_outer_idx, outer_poly in enumerate(perim_outer_geoms):
        outer_poly = outer_poly.buffer(0)
        spot_flag = not np.any(inter_matrix[:, poly_outer_idx])

        # --------------------------------------------------
        # Determine valid child polygons
        # --------------------------------------------------
        if spot_flag:
            # No intersecting children → compute distances to all children
            distances = [g.distance(outer_poly) for g in perim_inner_geoms]
            nearest_idx = np.argmin(distances)
            if distances[nearest_idx] > spot_threshold:
                continue
            polyids = [nearest_idx]
        else:
            polyids = [ii for ii in range(len(perim_inner_geoms))
                       if inter_matrix[ii, poly_outer_idx]]

        if not polyids:
            continue

        poly_best_dist = -np.inf
        poly_best_pair = None
        poly_best_poly = None
        poly_best_parent_idx = None

        # --------------------------------------------------
        # Iterate over child polygons
        # --------------------------------------------------
        for poly_inner_idx in polyids:
            child_poly = perim_inner_geoms[poly_inner_idx].buffer(0)

            if child_poly.is_empty:
                continue

            # Sample points along child perimeter
            n_child = max(1, int(child_poly.length * points_per_meter))
            if n_child == 0:
                # fallback: use coords from the polygon
                child_pts = [Point(c) for c in child_poly.exterior.coords]
            else:
                # sample_perimeter should return shapely Points
                child_pts = sample_perimeter(child_poly, n_child)

            if child_poly.intersects(outer_poly):
                # overlapping child → compute max distance from child points to parent exterior
                outer_boundary = outer_poly.exterior
                dists = [pt.distance(outer_boundary) for pt in child_pts]
                best_idx = np.argmax(dists)
                pt_child = np.array(child_pts[best_idx].coords[0])
                pt_parent = np.array(outer_boundary.interpolate(outer_boundary.project(child_pts[best_idx])).coords[0])
                max_dist = dists[best_idx]

                test_line = LineString([tuple(pt_child), tuple(pt_parent)])
                if test_line.length == 0:
                    continue

                sample_step = min(50, test_line.length)
                n_samples = max(2, int(math.ceil((test_line.length - sample_step) / 50)) + 1)
                sample_distances = np.linspace(sample_step, test_line.length, n_samples)

                seen_outside = False
                invalid_vector = False
                for sample_dist in sample_distances:
                    is_inside_parent = child_poly.covers(test_line.interpolate(sample_dist))
                    if not is_inside_parent:
                        seen_outside = True
                    elif seen_outside:
                        invalid_vector = True
                        break

                if invalid_vector:
                    continue

            else:
                # disconnected child → nearest points as usual
                pt_child_sh, pt_parent_sh = nearest_points(child_poly, outer_poly)
                pt_child = np.array([pt_child_sh.x, pt_child_sh.y])
                pt_parent = np.array([pt_parent_sh.x, pt_parent_sh.y])
                max_dist = np.linalg.norm(pt_parent - pt_child)

            if debug:
                logger.info(f"Poly_outer {poly_outer_idx}, Poly_inner {poly_inner_idx}, "
                    f"dist_val={max_dist:.2f}")
                logger.info(f"Child coord: {pt_child}, Parent coord: {pt_parent}")

            if max_dist > poly_best_dist:
                poly_best_dist = max_dist
                poly_best_pair = (pt_child, pt_parent)
                poly_best_poly = (child_poly, outer_poly)
                poly_best_parent_idx = poly_inner_idx

        # Append results if a valid vector was found
        if poly_best_pair is not None:
            result_dist.append(poly_best_dist)
            result_coord_pair.append(poly_best_pair)
            result_poly_pair.append(poly_best_poly)
            result_parent_idx.append(poly_best_parent_idx)

    # Global maximum across all parents
    if result_dist:
        max_loc = np.argmax(result_dist)
        return (
            result_dist[max_loc],
            result_coord_pair[max_loc][0],
            result_coord_pair[max_loc][1],
            result_parent_idx[max_loc]
        )
    else:
        return np.nan, None, None, None

def sample_perimeter(poly, n_points):
    length = poly.length
    if n_points <= 0:
        return []

    distances = np.linspace(0, length, n_points, endpoint=False)
    sampled_pts = [poly.exterior.interpolate(d) for d in distances]

    # Return as Shapely Points, not NumPy arrays
    return [Point(p.x, p.y) for p in sampled_pts]